# NB11 - Publish + dataset card

**Run once, last.** Writes the dataset card (`README.md`) describing schema, provenance, license caveats, generators, and the split; optionally flips the repo to public; and tags a version.

Set `MAKE_PUBLIC = True` only when you are satisfied with NB10. GPU not needed.

In [ ]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)
# diffusers stack + hub/io. sentencepiece+protobuf are needed by the T5 text encoders (Flux/PixArt).
pip("diffusers>=0.30", "transformers>=4.40", "accelerate", "safetensors",
    "sentencepiece", "protobuf", "huggingface_hub>=0.23", "pyarrow", "pillow")
print("deps installed")

In [ ]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"
MAKE_PUBLIC = False        # set True to flip the dataset public
VERSION_TAG = "v1.0"
import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN","HUGGINGFACE_TOKEN","HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)

In [ ]:
# gather final counts from the manifest (falls back to shard counts)
try:
    mp = C.hf_hub_download(REPO_ID, "manifest.parquet", repo_type="dataset", token=TOKEN)
    man = C.pq.read_table(mp, columns=["generator","label","split"])
    from collections import Counter
    by_gen = Counter(man.column("generator").to_pylist())
    by_split = Counter(man.column("split").to_pylist())
    n_real = sum(1 for l in man.column("label").to_pylist() if l == 0)
    n_ai = sum(1 for l in man.column("label").to_pylist() if l == 1)
except Exception as e:
    print("manifest missing, counting shards:", e)
    by_gen = {m: C.count_rows(REPO_ID, f"data/{m}/", TOKEN) for m in cfg["generators"]}
    n_real = C.count_rows(REPO_ID, "real/", TOKEN); n_ai = sum(by_gen.values()); by_split = {}
print("real:", n_real, "| ai:", n_ai, "| by gen:", dict(by_gen), "| by split:", dict(by_split))

In [ ]:
# build README.md (dataset card)
gens_md = "\n".join(f"| {m} | `{cfg['generators'][m]['model_id']}` | {cfg['generators'][m]['native']} | "
                     f"{cfg['generators'][m]['steps']} | {cfg['generators'][m]['guidance']} |"
                     for m in cfg["generators"])
card = f"""---
license: other
task_categories:
- image-classification
tags:
- ai-generated-image-detection
- synthetic-image-detection
size_categories:
- 100K<n<1M
---

# AI-Generated Image Detection Dataset

Paired real / AI images for training detectors. Every real image has one AI partner per
generator, and a pair shares a single image-grounded caption, so detectors are pushed toward
the synthesis fingerprint rather than scene content.

## Composition
- **Real:** {n_real} images (COCO + ImageNet), label `0`.
- **AI:** {n_ai} images across {len(cfg['generators'])} generators, label `1`.
- **Resolution:** {cfg['target_resolution']}x{cfg['target_resolution']} RGB PNG, identical canonical
  preprocessing for real and AI (pipeline version {cfg['pipeline_version']}).

## Generators
| key | model | native res | steps | guidance |
|-----|-------|-----------|-------|----------|
{gens_md}

## Pairing & captions
Captions come from **{cfg['caption_model']}** ({cfg['caption_task']}), enhanced once and capped to
{cfg['caption_max_tokens']} CLIP tokens, then reused byte-identically by every generator. `source_real_id`
links an AI image to its real counterpart (it equals `image_id` for real rows).

## Splits
Deterministic **pair-level** {int(cfg['split']['train']*100)}/{int(cfg['split']['val']*100)}/{int(cfg['split']['test']*100)}
split keyed on `source_real_id`, so a scene and all its AI partners share one split (no cross-split leakage).
See `manifest.parquet` (per-image: id, source_real_id, generator, label, sha256, split) and `splits.parquet`.

## Loading
```python
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
man = pq.read_table(hf_hub_download("{REPO_ID}", "manifest.parquet", repo_type="dataset"))
# image bytes live in real/*.parquet and data/<generator>/*.parquet, keyed by image_id
```

## Provenance, license & intended use
- **ImageNet** content is **non-commercial research only** (ILSVRC terms); **COCO** images are Flickr-sourced.
  This dataset inherits those terms: **non-commercial research use**. Not legal advice.
- AI images are synthetic outputs of the models above, each under its own license.
- Intended for training/evaluating AI-image detectors. See `validation_report.json` for the leak audit.

## Limitations
Core set is text-to-image (no img2img/reference conditioning). Detectors trained here target the
text-to-image threat model. Caption quality and per-model resolution choices bound realism.
"""
open("README.md","w").write(card)
C.HfApi(token=TOKEN).upload_file(path_or_fileobj="README.md", path_in_repo="README.md",
        repo_id=REPO_ID, repo_type="dataset", commit_message="dataset card")
print("README.md uploaded.")

In [ ]:
# optionally flip public + tag a version
api = C.HfApi(token=TOKEN)
if MAKE_PUBLIC:
    api.update_repo_visibility(repo_id=REPO_ID, private=False, repo_type="dataset")
    print("repo is now PUBLIC")
else:
    print("left private (set MAKE_PUBLIC=True to publish)")
try:
    api.create_tag(repo_id=REPO_ID, repo_type="dataset", tag=VERSION_TAG)
    print("tagged", VERSION_TAG)
except Exception as e:
    print("tag note:", e)
print("\nNB11 complete - dataset build finished.")